In [ ]:
# ============================================================
# Interactive Bank Data Analysis & Forecasting - Colab Version
# ============================================================
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jacksoncrow/stock-market-dataset")

print("Path to dataset files:", path)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)

# ============================================================
# 1. Upload CSV
# ============================================================

print("Upload your CSV file (bank data)")
uploaded = files.upload()  # Colab will prompt to upload

# Get uploaded file name
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
print(f"\nFile '{file_name}' loaded successfully!\n")

# ============================================================
# 2. Overview of Data
# ============================================================

print("Number of rows (excluding headers):", df.shape[0])
print("Number of columns:", df.shape[1])
print("Column Names:", df.columns.tolist())
print("\nNull values per column:\n", df.isnull().sum())
print("\nFirst 5 rows:\n", df.head())

# ============================================================
# 3. Data Cleaning & User Interaction
# ============================================================

# Clean column names
df.columns = df.columns.str.replace('[^A-Za-z0-9]+', '', regex=True)

# Convert date columns to datetime if possible
for col in df.columns:
    if 'date' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col])
        except:
            pass

# Replace null/garbage values interactively
print("\n--- Data Cleaning ---")
for col in df.columns:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        print(f"\nColumn '{col}' has {null_count} null values.")
        replacement = input(f"Enter replacement value for nulls in '{col}' (or leave blank to skip): ")
        if replacement:
            df[col].fillna(replacement, inplace=True)

# Remove duplicates if user wants
remove_dup = input("\nRemove duplicate rows? (yes/no): ").strip().lower()
if remove_dup == 'yes':
    df.drop_duplicates(inplace=True)
    print("Duplicates removed.")

print("\nData cleaning completed!")

# ============================================================
# 4. Statistical Analysis
# ============================================================

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    print("\n--- Statistical Summary ---")
    print(df[numeric_cols].describe())

    # Ask user which stats to display
    print("\nAvailable statistics: mean, median, mode, min, max, std")
    stats_choice = input("Enter comma-separated statistics to calculate (or 'all'): ").strip().lower()
    if stats_choice == 'all':
        print("\nFull statistics:\n", df[numeric_cols].describe())
    else:
        stats_list = [s.strip() for s in stats_choice.split(',')]
        for s in stats_list:
            if s == 'mean':
                print("\nMean:\n", df[numeric_cols].mean())
            elif s == 'median':
                print("\nMedian:\n", df[numeric_cols].median())
            elif s == 'mode':
                print("\nMode:\n", df[numeric_cols].mode().iloc[0])
            elif s == 'min':
                print("\nMin:\n", df[numeric_cols].min())
            elif s == 'max':
                print("\nMax:\n", df[numeric_cols].max())
            elif s == 'std':
                print("\nStandard Deviation:\n", df[numeric_cols].std())
else:
    print("No numeric columns found for analysis.")

# ============================================================
# 5. Trend Visualization (Multi-color, Clean)
# ============================================================

print("\n--- Trend Visualization ---")
plt.figure(figsize=(12,6))
colors = plt.cm.get_cmap('tab20', len(numeric_cols))

for i, col in enumerate(numeric_cols):
    plt.plot(df[col], label=col, color=colors(i))

plt.title('Trends of Numeric Columns')
plt.xlabel('Index')
plt.ylabel('Value')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

# ============================================================
# 6. Predictive Forecasting
# ============================================================

print("\n--- Predictive Forecasting ---")
forecast_cols = [c for c in numeric_cols if c.lower() not in ['id']]
if forecast_cols:
    print("Numeric columns available for forecasting:", forecast_cols)
    forecast_col = input("Select column for forecasting (or leave blank to skip): ").strip()

    if forecast_col and forecast_col in df.columns:
        print("\nForecasting Methods: 1. Linear Regression  2. ARIMA")
        method = input("Select forecasting method (1/2): ").strip()

        if method == '1':
            df_forecast = df[[forecast_col]].dropna()
            X = np.arange(len(df_forecast)).reshape(-1,1)
            y = df_forecast.values
            model = LinearRegression()
            model.fit(X,y)
            future_days = int(input("Enter number of future days to forecast: "))
            X_future = np.arange(len(df_forecast), len(df_forecast)+future_days).reshape(-1,1)
            y_pred = model.predict(X_future)
            print(f"\nForecasted values for next {future_days} days:\n", y_pred.flatten())
            plt.figure(figsize=(12,5))
            plt.plot(df_forecast.values, label='Original', color='blue')
            plt.plot(range(len(df_forecast), len(df_forecast)+future_days), y_pred, label='Forecast', linestyle='--', color='red')
            plt.title(f'Linear Regression Forecast for {forecast_col}')
            plt.legend()
            plt.tight_layout()
            plt.show()

        elif method == '2':
            series = df[forecast_col].dropna().astype(float)
            p = int(input("Enter AR term (p): "))
            d = int(input("Enter I term (d): "))
            q = int(input("Enter MA term (q): "))
            future_steps = int(input("Enter number of future steps to forecast: "))
            model = ARIMA(series, order=(p,d,q))
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=future_steps)
            print(f"\nForecasted values for next {future_steps} steps:\n", forecast)
            plt.figure(figsize=(12,5))
            plt.plot(series, label='Original', color='blue')
            plt.plot(range(len(series), len(series)+future_steps), forecast, label='Forecast', linestyle='--', color='red')
            plt.title(f'ARIMA Forecast for {forecast_col}')
            plt.legend()
            plt.tight_layout()
            plt.show()
        else:
            print("Invalid method selected. Skipping forecasting.")
    else:
        print("Forecasting skipped.")

# ============================================================
# 7. Save Cleaned Data (Excel Only)
# ============================================================

save_file = input("\nDo you want to save the cleaned dataset? (yes/no): ").strip().lower()
if save_file == 'yes':
    output_file = input("Enter output Excel file name (e.g., cleaned_data.xlsx): ").strip()
    if not output_file.endswith('.xlsx'):
        output_file += '.xlsx'
    df.to_excel(output_file, index=False)
    files.download(output_file)
    print(f"Cleaned data saved to {output_file}")

print("\n--- Analysis Completed! ---")


Using Colab cache for faster access to the 'stock-market-dataset' dataset.
Path to dataset files: /kaggle/input/stock-market-dataset
Upload your CSV file (bank data)
